<a href="https://colab.research.google.com/github/ChristopherMwanginjoroge/deep-learning/blob/main/fnn3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch

from torch.utils.data import DataLoader,TensorDataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim



housing = fetch_california_housing(as_frame=False)

X=housing.data
y=housing.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train = scaler.fit_transform(y_train.reshape(-1, 1))
y_test = scaler.transform(y_test.reshape(-1, 1))

#convert to pytorch tensors

X_train_tensor =torch.FloatTensor(X_train)
y_train_tensor=torch.FloatTensor(y_train)

X_test_tensor =torch.FloatTensor(X_test)
y_test_tensor=torch.FloatTensor(y_test)

train_dataset=TensorDataset(X_train_tensor,y_train_tensor)

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)


test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Training batches ready: {len(train_loader)}")
print("Notice: Our target values (y) are now small scaled numbers, not massive dollar amounts.")


Training batches ready: 516
Notice: Our target values (y) are now small scaled numbers, not massive dollar amounts.


In [5]:
class HousingModel(nn.Module):
  def __init__(self):
    super(HousingModel,self).__init__()

    self.fc1=nn.Linear(in_features=8,out_features=64)
    self.fc2=nn.Linear(in_features=64,out_features=32)
    self.fc3=nn.Linear(in_features=32,out_features=1)

    self.dropout=nn.Dropout(p=0.2)

    def forward(self,x):
      x=F.relu(self.fc1(x))
      x=self.dropout(x)
      x=F.relu(self.fc2(x))
      x=self.dropout(x)
      x=self.fc3(x)

      return x

model=HousingModel()
print(model)


HousingModel(
  (fc1): Linear(in_features=8, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)


In [6]:
criterion=nn.MSELoss()

optimizer=optim.Adam(model.parameters(),lr=0.001)

epochs=100


for epoch in range(epochs):
  model.train()
  running_mse=0.0


  for inputs,targets in train_loader:
    outputs = model(inputs)

    loss = criterion(outputs, targets)

    optimizer.zero_grad()

    loss.backward()
    optimizer.step()

    running_mse += loss.item()


  if (epoch+1) % 10 == 0:
    print(f"Epoch [{epoch+1}/{epochs}], MSE: {running_mse/len(train_loader):.4f}")




# testing

model.eval()
test_mse = 0.0
predictions_list = []
actuals_list = []


with torch.no_grad():
  for inputs,targets in test_loader:
    outputs=model(inputs)

    loss=criterion(outputs,targets)
    test_mse += loss.item()

    predictions_list.extend(outputs.squeeze().tolist())
    actuals_list.extend(targets.squeeze().tolist())

avg_test_mse = test_mse / len(test_loader)
print(f"\nFinal Test MSE (Scaled): {avg_test_mse:.4f}")


NotImplementedError: Module [HousingModel] is missing the required "forward" function